# 05_split_features: Track 1 split, feature sets and target check
Input: data/processed/itsm_clean.pkl. Outputs: src/project_config.py, src/features.py, data/processed/track1_train.pkl and track1_test.pkl. Aggregates only.
Each %%writefile cell must contain only the file contents; imports go in a separate cell.

## 1. Setup and load the clean data

In [1]:
import os, sys, importlib
import numpy as np
import pandas as pd
sys.path.append("../src")

df = pd.read_pickle("../data/processed/itsm_clean.pkl")

<details>
<summary><b>Code notes, line by line</b></summary>

- `import os, sys, importlib / numpy / pandas` : Libraries.
- `sys.path.append("../src")` : Lets the notebook import the project modules it writes.
- `df = pd.read_pickle("../data/processed/itsm_clean.pkl")` : The clean checkpoint from 03.

</details>

## 2. Split dates in one place (src/project_config.py)

In [2]:
%%writefile ../src/project_config.py
START = "2013-10-01"
TEST_START = "2014-02-01"
CUT = "2014-03-17"
HIGH = [1, 2]

Overwriting ../src/project_config.py


<details>
<summary><b>Code notes, line by line</b></summary>

- `%%writefile ../src/project_config.py` : Saves the split dates once, so every notebook uses the same values.
- `START = "2013-10-01"` : First day of the modelling window.
- `TEST_START = "2014-02-01"` : First day of the test period.
- `CUT = "2014-03-17"` : Cut-off: tickets opened on or after this date are left out (censored).
- `HIGH = [1, 2]` : Priorities counted as High.

</details>

## 3. Build the Track 1 split and add open-time features
Only tickets with a priority label. Train: opened 2013-10-01 to 2014-01-31. Test: 1 February to 16 March 2014. High = priority 1 or 2. Rows are sorted by Open_Time (time-ordered validation depends on it). `month` is deliberately not a feature: the test months never occur in training.

In [3]:
sys.modules.pop("project_config", None)
importlib.invalidate_caches()
from project_config import START, TEST_START, CUT, HIGH

mw = df[(df["Open_Time"] >= START) & (df["Open_Time"] < CUT) & df["Priority"].notna()]
mw = mw.sort_values("Open_Time").reset_index(drop=True)
mw["High"] = mw["Priority"].isin(HIGH).astype(int)
mw["hour"] = mw["Open_Time"].dt.hour
mw["weekday"] = mw["Open_Time"].dt.day_of_week

train = mw[mw["Open_Time"] < TEST_START]
test = mw[mw["Open_Time"] >= TEST_START]
print("Table 3. Track 1 split | train:", len(train), "tickets,", train["High"].sum(), "High | test:", len(test), "tickets,", test["High"].sum(), "High")

Table 3. Track 1 split | train: 30845 tickets, 541 High | test: 10229 tickets, 115 High


<details>
<summary><b>Code notes, line by line</b></summary>

- `sys.modules.pop("project_config", None); importlib.invalidate_caches()` : Forces the freshly written config to be imported.
- `from project_config import START, TEST_START, CUT, HIGH` : The shared settings.
- `mw = df[(Open_Time >= START) & (Open_Time < CUT) & Priority.notna()]` : Labelled tickets in the modelling window.
- `mw.sort_values("Open_Time").reset_index(drop=True)` : Time order: TimeSeriesSplit later assumes rows are in order.
- `mw["High"] = mw["Priority"].isin(HIGH).astype(int)` : The Track 1 target: 1 for P1 or P2.
- `mw["hour"], mw["weekday"]` : Opening hour (0-23) and day of week (0 = Monday), known at creation.
- `train = mw[Open_Time < TEST_START]; test = mw[Open_Time >= TEST_START]` : Time-based split: no future ticket is used for training.
- `print(len(train), train["High"].sum(), len(test), test["High"].sum())` : Sizes and High counts of both parts.

</details>

**Result.** Training has 30,845 labelled tickets with 541 High (1.75%), the test period 10,229 with 115 High (1.12%) (Table 3).

## 4. Categorical columns: values unseen in training

In [4]:
cats = ["CI_Name", "CI_Cat", "CI_Subcat", "WBS", "Category"]
print("Table 4. Values seen in training, and test rows with an unseen value")
for c in cats:
    unseen = (~test[c].isin(train[c].unique())).mean() * 100
    print(f"{c}: {train[c].nunique()} values in train, {unseen:.1f}% of test rows have a value never seen in train")

Table 4. Values seen in training, and test rows with an unseen value
CI_Name: 2471 values in train, 4.9% of test rows have a value never seen in train
CI_Cat: 13 values in train, 0.0% of test rows have a value never seen in train
CI_Subcat: 57 values in train, 0.0% of test rows have a value never seen in train
WBS: 255 values in train, 0.6% of test rows have a value never seen in train
Category: 4 values in train, 0.0% of test rows have a value never seen in train


<details>
<summary><b>Code notes, line by line</b></summary>

- `for c in cats:` : Each categorical column.
- `(~test[c].isin(train[c].unique())).mean() * 100` : Share of test rows whose value never occurs in training: the encoder must handle these (handle_unknown).

</details>

**Result.** CI_Cat, CI_Subcat and Category have no unseen values in the test period, WBS 0.6% of test rows and CI_Name 4.9% (Table 4). The encoder's `handle_unknown` setting therefore matters mainly for CI_Name, which is not a headline feature.

## 5. Baseline reference and the saved train and test files

In [5]:
print("Table 5. Reference points for the test period")
print("Test High rate (a random model's average precision):", round(test["High"].mean(), 4))
print("Accuracy of always predicting not-High:", round(1 - test["High"].mean(), 4))

os.makedirs("../data/processed", exist_ok=True)
train.to_pickle("../data/processed/track1_train.pkl")
test.to_pickle("../data/processed/track1_test.pkl")

Table 5. Reference points for the test period
Test High rate (a random model's average precision): 0.0112
Accuracy of always predicting not-High: 0.9888


<details>
<summary><b>Code notes, line by line</b></summary>

- `test["High"].mean()` : The test High rate, which is also the average precision a random model would get.
- `1 - test["High"].mean()` : Accuracy of always predicting not-High: shows why accuracy is a poor measure here.
- `train.to_pickle(...); test.to_pickle(...)` : Saves the split for 06 and 09.

</details>

**Result.** A random model's average precision is 0.0112, and always predicting not-High is 98.88% accurate (Table 5). A useful model must beat the first by a wide margin; the second shows why accuracy is not used.

## 6. Do the features carry signal? (aggregates only)

In [6]:
for t, c in zip("ab", ["Category", "CI_Cat"]):
    print(f"Table 6{t}. High rate by {c} (training months, 8 largest)")
    g = train.groupby(c)["High"].agg(rows="size", high="sum")
    g["rate_%"] = (g["high"] / g["rows"] * 100).round(2)
    print(g.sort_values("rows", ascending=False).head(8), "\n")

g = train.groupby("CI_Name")["High"].agg(rows="size", high="sum")
big = g["rows"] >= 30
print("Table 6c. High tickets on frequent (30+ rows) and rare CIs (training months)")
print("CI_Names with at least one High:", int((g["high"] > 0).sum()), "of", len(g))
print("CI_Names with 30+ train rows:", int(big.sum()))
print("Share of train rows on those CIs:", round(g.loc[big, "rows"].sum() / g["rows"].sum() * 100, 1), "%")
print("Share of train High on those CIs:", round(g.loc[big, "high"].sum() / g["high"].sum() * 100, 1), "%")

print("Table 6d. High rate (%) by opening hour and by weekday (0 = Monday), training months")
print("High rate % by hour:", train.groupby("hour")["High"].mean().mul(100).round(1).to_dict())
print("High rate % by weekday:", train.groupby("weekday")["High"].mean().mul(100).round(1).to_dict())

Table 6a. High rate by Category (training months, 8 largest)
                          rows  high  rate_%
Category                                    
incident                 24416   537    2.20
request for information   6421     4    0.06
complaint                    7     0    0.00
request for change           1     0    0.00 

Table 6b. High rate by CI_Cat (training months, 8 largest)
                    rows  high  rate_%
CI_Cat                                
application        21779   143    0.66
subapplication      5178    23    0.44
computer            2516   345   13.71
storage              424     5    1.18
hardware             255     2    0.78
software             200     1    0.50
displaydevice        143     0    0.00
officeelectronics    106     0    0.00 

Table 6c. High tickets on frequent (30+ rows) and rare CIs (training months)
CI_Names with at least one High: 358 of 2471
CI_Names with 30+ train rows: 139
Share of train rows on those CIs: 83.5 %
Share of train High

<details>
<summary><b>Code notes, line by line</b></summary>

- `for c in ["Category", "CI_Cat"]: groupby(c)["High"].agg(rows="size", high="sum")` : Tickets and High tickets per category, with the High rate.
- `g = train.groupby("CI_Name")["High"].agg(...); big = g["rows"] >= 30` : Per CI: tickets and High; 'big' marks CIs with 30+ training rows.
- `print(... int((g["high"] > 0).sum()) ...)` : How many CIs ever had a High ticket.
- `Share of train rows / High on those CIs` : Whether High tickets sit on the frequent CIs or on rare ones.
- `groupby("hour")["High"].mean(); groupby("weekday")["High"].mean()` : High rate by opening hour and by weekday.

</details>

### Findings
- **Category (Table 6a):** incidents have a 2.20% High rate (537 of 24,416), requests for information 0.06% (4 of 6,421).
- **CI_Cat (Table 6b):** computer tickets have a 13.71% High rate (345 of 2,516, which is 64% of all High tickets in training), against 0% to 1.18% for the other asset types.
- **CI_Name (Table 6c):** 139 of 2,471 CIs have 30 or more training rows. They hold 83.5% of tickets but only 23.7% of High tickets, so most High tickets sit on rare CIs; only 358 CIs ever had a High ticket.
- **Hour and weekday (Table 6d):** the High rate is 23.8% to 33.3% at hours 4 to 6 and 8.8% (Saturday) and 19.4% (Sunday) at weekends, against 1.4% to 2.1% on weekdays (some hours have small counts).

## 7. Feature sets (src/features.py)
Every column is given one role, so the models only see fields that are known **when a ticket is opened**.

| Set | Columns | Used for |
|---|---|---|
| **Set B** (headline) | CI_Cat, CI_Subcat, WBS, Category, hour, weekday | Every model in Tracks 1, 3 and 4 (Track 4 adds CI_Name) |
| **Set A** (contrast only) | Set B + Impact and Urgency | Only `06_track1_models`, section 5, to show leakage |

**Why two sets.** Priority is set by a fixed lookup of Impact and Urgency: no Impact x Urgency pair maps to more than one priority (`02_ingest_and_profile`, Table 8b), and High is almost exactly "Impact 1 or 2" (Table 10c below). A model given Impact and Urgency (Set A) just relearns that lookup and scores almost perfectly, but it predicts nothing useful: once those two fields are filled in, the priority is already decided. Set B leaves them out and asks the real question: can a High-priority ticket be recognised from what it is about (the asset) and when it arrived? Set A is reported only to make this leakage visible (`06`, Tables 5a and 5b: CV average precision 0.994 to 1.000).

**Never used as features:** post-event fields (known only after handling), identifiers, and the six fields whose timing is unconfirmed (tested only as a sensitivity check in `06`, section 18). Section 8 checks that every column has a role, and section 9 asserts that none of the excluded ones reaches X.

In [7]:
%%writefile ../src/features.py
asset_cols = ["CI_Cat", "CI_Subcat", "WBS", "Category"]
ci_name    = ["CI_Name"]
time_feats = ["hour", "weekday"]
matrix_cols = ["Impact", "Urgency"]

post_event = ["Status", "Resolved_Time", "Close_Time", "Reopen_Time", "Has_Reopen_Time",
              "Handle_Time_hrs", "Elapsed_hrs", "Closure_Code"]
unsure_timing = ["No_of_Reassignments", "No_of_Related_Interactions", "No_of_Related_Incidents",
                 "No_of_Related_Changes", "Has_Related_Change", "KB_number"]
identifiers = ["Incident_ID", "Related_Interaction", "Related_Change"]

set_a = asset_cols + time_feats + matrix_cols   # includes Impact and Urgency
set_b = asset_cols + time_feats                 # the honest test

Overwriting ../src/features.py


<details>
<summary><b>Code notes, line by line</b></summary>

- `%%writefile ../src/features.py` : Saves the feature lists in one module used by 06, 08 and 09.
- `asset_cols` : Asset fields known at creation: CI_Cat, CI_Subcat, WBS, Category.
- `ci_name` : The individual asset, kept separate (used only in sensitivity checks and Track 4).
- `time_feats` : Opening hour and weekday.
- `matrix_cols` : Impact and Urgency, which determine priority by lookup.
- `post_event` : Fields only known after the ticket is handled: never features.
- `unsure_timing` : Fields that may be filled after creation: excluded from the headline, tested in 06 section 18.
- `identifiers` : IDs: never features.
- `set_a = asset_cols + time_feats + matrix_cols` : Set A, the leakage contrast.
- `set_b = asset_cols + time_feats` : Set B, the honest headline feature set.

</details>

## 8. Check that every column is classified

In [8]:
sys.modules.pop("features", None)
importlib.invalidate_caches()
from features import *

known = set(asset_cols + ci_name + time_feats + matrix_cols + post_event
            + unsure_timing + identifiers + ["Priority", "High", "Open_Time"])
print("Table 8. Columns not assigned to any group (should be empty):", sorted(set(train.columns) - known))

Table 8. Columns not assigned to any group (should be empty): []


<details>
<summary><b>Code notes, line by line</b></summary>

- `sys.modules.pop("features", None); from features import *` : Imports the fresh feature lists.
- `known = set(asset_cols + ... + ["Priority", "High", "Open_Time"])` : Every column that has been assigned a role.
- `print(sorted(set(train.columns) - known))` : Should print []: no column is left unclassified.

</details>

## 9. Explicit X and y (Set B features)

In [9]:
X_train, y_train = train[set_b].astype(str), train["High"]
X_test,  y_test  = test[set_b].astype(str),  test["High"]
print("Table 9. X and y: shapes, High counts and features")
print(X_train.shape, X_test.shape, int(y_train.sum()), int(y_test.sum()))

assert list(X_train.columns) == list(X_test.columns) == set_b
assert not set(X_train.columns) & set(post_event + unsure_timing + identifiers + matrix_cols + ci_name)
assert set(y_train.unique()) <= {0, 1} and set(y_test.unique()) <= {0, 1}
print("features:", list(X_train.columns))

Table 9. X and y: shapes, High counts and features
(30845, 6) (10229, 6) 541 115
features: ['CI_Cat', 'CI_Subcat', 'WBS', 'Category', 'hour', 'weekday']


<details>
<summary><b>Code notes, line by line</b></summary>

- `X_train, y_train = train[set_b].astype(str), train["High"]` : Features as text (so hour and weekday are treated as categories) and the target.
- `X_test, y_test = ...` : The same for the test period.
- `print(X_train.shape, ...)` : Sizes and High counts.
- `assert list(X_train.columns) == list(X_test.columns) == set_b` : Train and test have the same columns in the same order.
- `assert not set(X_train.columns) & set(post_event + ...)` : None of the excluded columns slipped into the features.
- `assert set(y_train.unique()) <= {0, 1}` : The target is binary.

</details>

**Result.** No column is left unclassified (Table 8 is empty), and X holds only the six Set B features with 541 and 115 High tickets (Table 9); the assertions confirm that no excluded field reaches the model.

## 10. Target check: High against Priority and Impact

In [10]:
print("Table 10a. High against Priority (training months)")
print(pd.crosstab(train["High"], train["Priority"]))
print("Table 10b. Class counts, train and test")
print(y_train.value_counts().to_dict(), y_test.value_counts().to_dict())
print("Table 10c. Impact against High (modelling window)")
print(pd.crosstab(mw["Impact"], mw["High"]))

Table 10a. High against Priority (training months)
Priority  2.0   3.0    4.0    5.0
High                             
0           0  3517  15180  11607
1         541     0      0      0
Table 10b. Class counts, train and test
{0: 30304, 1: 541} {0: 10114, 1: 115}
Table 10c. Impact against High (modelling window)
High        0    1
Impact            
1.0         0    3
2.0         0  649
3.0      4679    4
4.0     20378    0
5.0     15361    0


<details>
<summary><b>Code notes, line by line</b></summary>

- `pd.crosstab(train["High"], train["Priority"])` : Checks High = 1 exactly for priorities 1 and 2.
- `y_train.value_counts(), y_test.value_counts()` : Class counts in both periods.
- `pd.crosstab(mw["Impact"], mw["High"])` : High against Impact: shows High is almost 'Impact 1 or 2', which is why Impact is excluded.

</details>

### Findings
- High is priority 1 or 2 (Table 10a). In the training period every High ticket is priority 2 (541); the 3 priority-1 tickets in the window all fall in the test period. Train has 30,304 not-High and 541 High tickets, test 10,114 and 115 (Table 10b).
- High is almost "Impact 1 or 2" (Table 10c): of the 656 High tickets in the modelling window, 652 have Impact 1 or 2 (3 + 649) and 4 have Impact 3. Priority is a lookup of Impact and Urgency, which is why Impact and Urgency are excluded from the headline features.